# Fusion Retrieval (BM25 + Vectors)

Combine keyword and semantic search - one of the most useful upgrades in practice.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** `pip install rank_bm25`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

tokenize = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tokenize(c) for c in chunks])

def dense_scores(q): return store.vecs @ embed(q)[0]
def bm25_scores(q):  return np.array(bm25.get_scores(tokenize(q)))

### Option A: weighted score fusion
Min-max normalise both score lists, then mix with weight `alpha`.

In [ ]:
def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-9)

def weighted_fusion(q, k=4, alpha=0.5):
    scores = alpha * norm(dense_scores(q)) + (1 - alpha) * norm(bm25_scores(q))
    return [{"idx": int(i), "text": chunks[i], "score": float(scores[i]), "meta": {}} for i in np.argsort(-scores)[:k]]

### Option B: Reciprocal Rank Fusion
Scores from different systems are not comparable, but ranks are. Each list adds `1 / (60 + rank)`.

In [ ]:
def rrf(q, k=4, c=60):
    fused = {}
    for scores in (dense_scores(q), bm25_scores(q)):
        for rank, i in enumerate(np.argsort(-scores)):
            fused[int(i)] = fused.get(int(i), 0) + 1 / (c + rank + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [{"idx": i, "text": chunks[i], "score": fused[i], "meta": {}} for i in top]

query = "cross-encoder/ms-marco-MiniLM-L-6-v2"      # an exact-name query that embeddings handle poorly
for name, fn in [("dense", lambda q, k: store.search(q, k)), ("weighted", weighted_fusion), ("rrf", rrf)]:
    print("---", name); show(fn(query, 3), width=100)

In [ ]:
print(answer("What is the model name of the small widely used reranker?", [h["text"] for h in rrf("small widely used reranker model")]))